# Advanced Multimodal Applications

**Module 12: Multimodal AI**  
**Objective**: Building production multimodal systems

Advanced Topics:
- Vision-language models (LLaVA, GPT-4V)
- Image captioning
- Visual Question Answering (VQA)
- Multimodal RAG
- Vision-language instruction tuning

## What You'll Learn
1. LLaVA architecture (vision + LLM)
2. Image captioning models
3. VQA systems
4. Multimodal RAG for documents
5. Vision-language instruction following
6. Production multimodal pipelines

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
np.random.seed(42)

## 1. LLaVA: Large Language and Vision Assistant

**LLaVA** connects vision encoder to LLM for visual instruction following.

**Architecture**:
```
Image → Vision Encoder → Projection → LLM → Text Response
                (CLIP)     (Linear)   (Vicuna/LLaMA)
```

**Key Innovation**: Train projection layer to map vision features to LLM token space

**Training stages**:
1. **Pre-training**: Align vision and language (projection only)
2. **Fine-tuning**: Instruction following (full model or LoRA)

In [ ]:
class VisionEncoder(nn.Module):
    """
    Vision encoder (simplified CLIP image encoder).
    """
    
    def __init__(self, embed_dim: int = 768):
        super().__init__()
        
        # Simplified Vision Transformer-like encoder
        self.conv_proj = nn.Conv2d(3, embed_dim, kernel_size=16, stride=16)
        
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=12,
                dim_feedforward=3072,
                dropout=0.1,
                batch_first=True
            ),
            num_layers=12
        )
        
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: (batch_size, 3, H, W)
            
        Returns:
            features: (batch_size, num_patches, embed_dim)
        """
        # Patchify: (B, 3, 224, 224) → (B, 768, 14, 14)
        x = self.conv_proj(images)
        
        # Flatten patches: (B, 768, 14, 14) → (B, 196, 768)
        batch_size = x.size(0)
        x = x.flatten(2).transpose(1, 2)
        
        # Transform
        features = self.transformer(x)  # (B, 196, 768)
        
        return features

class VisionLanguageProjector(nn.Module):
    """
    Projects vision features to LLM token embedding space.
    """
    
    def __init__(self, vision_dim: int = 768, llm_dim: int = 4096):
        super().__init__()
        
        # Multi-layer projection
        self.projection = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        )
        
    def forward(self, vision_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            vision_features: (batch_size, num_patches, vision_dim)
            
        Returns:
            llm_features: (batch_size, num_patches, llm_dim)
        """
        return self.projection(vision_features)

class SimpleLLM(nn.Module):
    """
    Simplified LLM for demonstration (real: LLaMA, Vicuna).
    """
    
    def __init__(self, vocab_size: int = 32000, d_model: int = 4096, 
                 n_layers: int = 32, n_heads: int = 32):
        super().__init__()
        
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Parameter(torch.randn(2048, d_model))
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        
        self.lm_head = nn.Linear(d_model, vocab_size)
        
    def forward(self, input_ids: torch.Tensor, 
                vision_embeds: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            input_ids: (batch_size, seq_len)
            vision_embeds: (batch_size, num_patches, d_model) optional vision features
            
        Returns:
            logits: (batch_size, seq_len, vocab_size)
        """
        # Text embeddings
        seq_len = input_ids.size(1)
        text_embeds = self.token_embedding(input_ids)  # (B, seq_len, d_model)
        text_embeds = text_embeds + self.position_embedding[:seq_len]
        
        # Concatenate vision embeddings if provided
        if vision_embeds is not None:
            # Vision tokens come first
            combined = torch.cat([vision_embeds, text_embeds], dim=1)
            seq_len = combined.size(1)
        else:
            combined = text_embeds
        
        # Causal mask
        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(device)
        
        # Transform
        # Note: For demo, using decoder without memory (self-attention only)
        output = self.transformer(combined, combined, tgt_mask=causal_mask)
        
        # Only return logits for text tokens (skip vision tokens)
        if vision_embeds is not None:
            num_vision_tokens = vision_embeds.size(1)
            output = output[:, num_vision_tokens:, :]
        
        logits = self.lm_head(output)
        
        return logits

class LLaVAModel(nn.Module):
    """
    LLaVA: Large Language and Vision Assistant.
    
    Combines vision encoder, projection, and LLM.
    """
    
    def __init__(self, vision_dim: int = 768, llm_dim: int = 4096, vocab_size: int = 32000):
        super().__init__()
        
        self.vision_encoder = VisionEncoder(vision_dim)
        self.projector = VisionLanguageProjector(vision_dim, llm_dim)
        self.llm = SimpleLLM(vocab_size, llm_dim)
        
    def forward(self, images: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: (batch_size, 3, H, W)
            input_ids: (batch_size, seq_len) text tokens
            
        Returns:
            logits: (batch_size, seq_len, vocab_size)
        """
        # Encode vision
        vision_features = self.vision_encoder(images)  # (B, num_patches, vision_dim)
        
        # Project to LLM space
        vision_embeds = self.projector(vision_features)  # (B, num_patches, llm_dim)
        
        # Generate with LLM
        logits = self.llm(input_ids, vision_embeds)
        
        return logits
    
    def generate(self, images: torch.Tensor, prompt_ids: torch.Tensor, 
                 max_length: int = 100) -> torch.Tensor:
        """
        Generate text from image and prompt.
        
        Args:
            images: (batch_size, 3, H, W)
            prompt_ids: (batch_size, prompt_len)
            max_length: Maximum generation length
            
        Returns:
            generated_ids: (batch_size, max_length)
        """
        self.eval()
        
        with torch.no_grad():
            # Encode vision
            vision_features = self.vision_encoder(images)
            vision_embeds = self.projector(vision_features)
            
            generated = prompt_ids.clone()
            
            for _ in range(max_length):
                # Get logits
                logits = self.llm(generated, vision_embeds)
                
                # Get next token (greedy)
                next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
                
                # Append
                generated = torch.cat([generated, next_token], dim=1)
                
                # Stop if EOS token (assuming token 2 is EOS)
                if (next_token == 2).all():
                    break
        
        return generated

# Create LLaVA model
llava = LLaVAModel(vision_dim=768, llm_dim=4096, vocab_size=32000).to(device)

total_params = sum(p.numel() for p in llava.parameters())
vision_params = sum(p.numel() for p in llava.vision_encoder.parameters())
projector_params = sum(p.numel() for p in llava.projector.parameters())
llm_params = sum(p.numel() for p in llava.llm.parameters())

print("LLaVA Model Architecture:")
print("="*70)
print(f"Total parameters: {total_params:,}")
print(f"  Vision encoder: {vision_params:,}")
print(f"  Projector: {projector_params:,} (trainable in stage 1)")
print(f"  LLM: {llm_params:,}")

# Test forward pass
batch_size = 2
test_images = torch.randn(batch_size, 3, 224, 224).to(device)
test_prompt = torch.randint(0, 32000, (batch_size, 20)).to(device)

with torch.no_grad():
    logits = llava(test_images, test_prompt)

print(f"\nForward pass:")
print(f"  Input images: {test_images.shape}")
print(f"  Input prompt: {test_prompt.shape}")
print(f"  Output logits: {logits.shape}")
print("\n✓ LLaVA processes images and text together!")

## 2. Image Captioning

Generate natural language descriptions of images.

**Architecture**:
- **Encoder**: Vision model (CNN, ViT)
- **Decoder**: Language model (LSTM, Transformer)
- **Training**: Cross-entropy on caption tokens

In [ ]:
class ImageCaptioningModel(nn.Module):
    """
    Image captioning with encoder-decoder architecture.
    """
    
    def __init__(self, vision_dim: int = 2048, embed_dim: int = 512, 
                 vocab_size: int = 10000, max_len: int = 50):
        super().__init__()
        
        # Vision encoder (simplified ResNet-like)
        self.vision_encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.vision_proj = nn.Linear(256, embed_dim)
        
        # Language decoder
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Parameter(torch.randn(max_len, embed_dim))
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=8,
            dim_feedforward=2048,
            dropout=0.1,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=6)
        
        self.output_proj = nn.Linear(embed_dim, vocab_size)
        
    def forward(self, images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: (batch_size, 3, H, W)
            captions: (batch_size, seq_len) caption tokens
            
        Returns:
            logits: (batch_size, seq_len, vocab_size)
        """
        # Encode image
        vision_features = self.vision_encoder(images)  # (B, 256, 1, 1)
        vision_features = vision_features.flatten(1)  # (B, 256)
        vision_embeds = self.vision_proj(vision_features).unsqueeze(1)  # (B, 1, embed_dim)
        
        # Embed captions
        seq_len = captions.size(1)
        caption_embeds = self.token_embedding(captions)  # (B, seq_len, embed_dim)
        caption_embeds = caption_embeds + self.position_embedding[:seq_len]
        
        # Causal mask
        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(device)
        
        # Decode
        output = self.decoder(caption_embeds, vision_embeds, tgt_mask=causal_mask)
        
        logits = self.output_proj(output)
        
        return logits
    
    def generate_caption(self, image: torch.Tensor, max_length: int = 20, 
                        start_token: int = 1, end_token: int = 2) -> List[int]:
        """
        Generate caption for image.
        
        Args:
            image: (3, H, W)
            max_length: Maximum caption length
            start_token: Start-of-sequence token
            end_token: End-of-sequence token
            
        Returns:
            caption_tokens: List of token IDs
        """
        self.eval()
        
        with torch.no_grad():
            # Encode image
            image_batch = image.unsqueeze(0).to(device)
            vision_features = self.vision_encoder(image_batch)
            vision_features = vision_features.flatten(1)
            vision_embeds = self.vision_proj(vision_features).unsqueeze(1)
            
            # Start with start token
            generated = torch.tensor([[start_token]]).to(device)
            
            for _ in range(max_length):
                # Embed current sequence
                caption_embeds = self.token_embedding(generated)
                caption_embeds = caption_embeds + self.position_embedding[:generated.size(1)]
                
                # Decode
                seq_len = generated.size(1)
                causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(device)
                output = self.decoder(caption_embeds, vision_embeds, tgt_mask=causal_mask)
                
                # Get next token logits
                logits = self.output_proj(output[:, -1, :])
                next_token = logits.argmax(dim=-1, keepdim=True)
                
                # Append
                generated = torch.cat([generated, next_token], dim=1)
                
                # Stop at end token
                if next_token.item() == end_token:
                    break
            
            return generated.squeeze(0).tolist()

# Create captioning model
captioner = ImageCaptioningModel(embed_dim=512, vocab_size=10000).to(device)

total_params = sum(p.numel() for p in captioner.parameters())
print("Image Captioning Model:")
print("="*70)
print(f"Total parameters: {total_params:,}")

# Test
test_image = torch.randn(3, 224, 224)
caption_tokens = captioner.generate_caption(test_image, max_length=15)

print(f"\nGenerated caption tokens: {caption_tokens}")

# Simulate vocabulary for display
vocab = {i: f"word_{i}" for i in range(10000)}
vocab[1] = "<START>"
vocab[2] = "<END>"

caption_words = [vocab.get(token, f"unk_{token}") for token in caption_tokens]
print(f"Caption: {' '.join(caption_words)}")

## 3. Visual Question Answering (VQA)

Answer natural language questions about images.

**Input**: Image + Question  
**Output**: Answer

**Approaches**:
1. **Classification**: Predict answer from fixed set
2. **Generation**: Generate free-form answer

In [ ]:
class VQAModel(nn.Module):
    """
    Visual Question Answering model.
    
    Combines image and question to predict answer.
    """
    
    def __init__(self, vision_dim: int = 2048, text_dim: int = 512, 
                 hidden_dim: int = 1024, num_answers: int = 1000):
        super().__init__()
        
        # Vision encoder
        self.vision_encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.vision_proj = nn.Linear(128, hidden_dim)
        
        # Question encoder
        self.question_embedding = nn.Embedding(10000, text_dim)
        self.question_lstm = nn.LSTM(text_dim, text_dim, batch_first=True)
        self.question_proj = nn.Linear(text_dim, hidden_dim)
        
        # Fusion and classification
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        
        self.classifier = nn.Linear(hidden_dim, num_answers)
        
    def forward(self, images: torch.Tensor, questions: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: (batch_size, 3, H, W)
            questions: (batch_size, seq_len)
            
        Returns:
            logits: (batch_size, num_answers)
        """
        # Encode image
        vision_features = self.vision_encoder(images)  # (B, 128, 1, 1)
        vision_features = vision_features.flatten(1)  # (B, 128)
        vision_embeds = self.vision_proj(vision_features)  # (B, hidden_dim)
        
        # Encode question
        question_embeds = self.question_embedding(questions)  # (B, seq_len, text_dim)
        _, (question_hidden, _) = self.question_lstm(question_embeds)
        question_hidden = question_hidden.squeeze(0)  # (B, text_dim)
        question_embeds = self.question_proj(question_hidden)  # (B, hidden_dim)
        
        # Fuse
        combined = torch.cat([vision_embeds, question_embeds], dim=1)  # (B, hidden_dim*2)
        fused = self.fusion(combined)  # (B, hidden_dim)
        
        # Classify
        logits = self.classifier(fused)  # (B, num_answers)
        
        return logits

# Create VQA model
vqa_model = VQAModel(hidden_dim=1024, num_answers=1000).to(device)

total_params = sum(p.numel() for p in vqa_model.parameters())
print("VQA Model:")
print("="*70)
print(f"Total parameters: {total_params:,}")

# Test
batch_size = 4
test_images = torch.randn(batch_size, 3, 224, 224).to(device)
test_questions = torch.randint(0, 10000, (batch_size, 15)).to(device)

with torch.no_grad():
    answer_logits = vqa_model(test_images, test_questions)

print(f"\nForward pass:")
print(f"  Images: {test_images.shape}")
print(f"  Questions: {test_questions.shape}")
print(f"  Answer logits: {answer_logits.shape}")

# Get top-3 answers
top_k = 3
top_probs, top_indices = F.softmax(answer_logits, dim=1).topk(top_k, dim=1)

print(f"\nExample predictions:")
for i in range(min(2, batch_size)):
    print(f"\nSample {i+1}:")
    for rank, (idx, prob) in enumerate(zip(top_indices[i], top_probs[i]), 1):
        print(f"  {rank}. Answer_{idx.item()} (prob: {prob.item():.4f})")

## 4. Multimodal RAG

Extend RAG to handle images and text together.

**Use cases**:
- Document QA with diagrams/charts
- Product search with images
- Scientific paper analysis

**Architecture**:
- Embed text and images separately
- Store in unified vector DB
- Retrieve relevant multimodal context
- Generate answer with vision-language model

In [ ]:
class MultimodalRAG:
    """
    Multimodal RAG system for documents with text and images.
    """
    
    def __init__(self, vision_encoder, text_encoder):
        self.vision_encoder = vision_encoder
        self.text_encoder = text_encoder
        
        self.image_embeddings = []
        self.text_embeddings = []
        self.image_metadata = []
        self.text_metadata = []
        
    def add_image(self, image: torch.Tensor, metadata: Dict):
        """Add image to index."""
        with torch.no_grad():
            if image.dim() == 3:
                image = image.unsqueeze(0)
            
            embedding = self.vision_encoder(image.to(device))
            
            self.image_embeddings.append(embedding.cpu())
            self.image_metadata.append(metadata)
    
    def add_text(self, text_tokens: torch.Tensor, metadata: Dict):
        """Add text to index."""
        with torch.no_grad():
            if text_tokens.dim() == 1:
                text_tokens = text_tokens.unsqueeze(0)
            
            embedding = self.text_encoder(text_tokens.to(device))
            
            self.text_embeddings.append(embedding.cpu())
            self.text_metadata.append(metadata)
    
    def search_images(self, query_text: torch.Tensor, top_k: int = 3) -> List[Dict]:
        """Search images using text query."""
        with torch.no_grad():
            if query_text.dim() == 1:
                query_text = query_text.unsqueeze(0)
            
            query_embed = self.text_encoder(query_text.to(device))
            
            # Compute similarities
            image_embeds = torch.cat(self.image_embeddings, dim=0).to(device)
            similarities = (query_embed @ image_embeds.t()).squeeze(0)
            
            # Get top-k
            top_indices = similarities.topk(min(top_k, len(self.image_embeddings))).indices.cpu()
            
            results = []
            for idx in top_indices:
                results.append({
                    'type': 'image',
                    'metadata': self.image_metadata[idx.item()],
                    'similarity': similarities[idx].item()
                })
            
            return results
    
    def search_text(self, query_text: torch.Tensor, top_k: int = 3) -> List[Dict]:
        """Search text using text query."""
        with torch.no_grad():
            if query_text.dim() == 1:
                query_text = query_text.unsqueeze(0)
            
            query_embed = self.text_encoder(query_text.to(device))
            
            # Compute similarities
            text_embeds = torch.cat(self.text_embeddings, dim=0).to(device)
            similarities = (query_embed @ text_embeds.t()).squeeze(0)
            
            # Get top-k
            top_indices = similarities.topk(min(top_k, len(self.text_embeddings))).indices.cpu()
            
            results = []
            for idx in top_indices:
                results.append({
                    'type': 'text',
                    'metadata': self.text_metadata[idx.item()],
                    'similarity': similarities[idx].item()
                })
            
            return results
    
    def search_multimodal(self, query_text: torch.Tensor, top_k: int = 5) -> List[Dict]:
        """Search both images and text."""
        # Search both modalities
        image_results = self.search_images(query_text, top_k=top_k)
        text_results = self.search_text(query_text, top_k=top_k)
        
        # Combine and sort by similarity
        all_results = image_results + text_results
        all_results.sort(key=lambda x: x['similarity'], reverse=True)
        
        return all_results[:top_k]

# Create multimodal RAG system
print("Multimodal RAG System")
print("="*70)

# Use simplified encoders for demo
class SimpleImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 512)
        )
    
    def forward(self, x):
        return F.normalize(self.encoder(x), p=2, dim=1)

class SimpleTextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(10000, 512)
        self.proj = nn.Linear(512, 512)
    
    def forward(self, x):
        embeds = self.embedding(x).mean(dim=1)
        return F.normalize(self.proj(embeds), p=2, dim=1)

image_encoder = SimpleImageEncoder().to(device).eval()
text_encoder = SimpleTextEncoder().to(device).eval()

multimodal_rag = MultimodalRAG(image_encoder, text_encoder)

# Add some documents
print("\nIndexing documents...")
for i in range(5):
    # Add images
    fake_image = torch.randn(3, 224, 224)
    multimodal_rag.add_image(fake_image, {
        'id': f'image_{i}',
        'caption': f'This is image {i}',
        'source': f'document_{i//2}.pdf'
    })
    
    # Add text
    fake_text = torch.randint(0, 10000, (50,))
    multimodal_rag.add_text(fake_text, {
        'id': f'text_{i}',
        'content': f'This is text chunk {i}',
        'source': f'document_{i//2}.pdf'
    })

print(f"✓ Indexed {len(multimodal_rag.image_embeddings)} images")
print(f"✓ Indexed {len(multimodal_rag.text_embeddings)} text chunks")

# Search
query = torch.randint(0, 10000, (10,))
results = multimodal_rag.search_multimodal(query, top_k=5)

print("\nSearch Results:")
print("-"*70)
for i, result in enumerate(results, 1):
    print(f"\n{i}. Type: {result['type']}")
    print(f"   Similarity: {result['similarity']:.4f}")
    print(f"   Metadata: {result['metadata']}")

print("\n✓ Multimodal RAG can retrieve both images and text!")

## 5. Vision-Language Instruction Tuning

Train models to follow vision-language instructions.

**Examples**:
- "Describe this image in detail"
- "What color is the car?"
- "Count the number of people"
- "Is there a cat in the image?"

**Training data format**:
```json
{
  "image": "path/to/image.jpg",
  "conversations": [
    {
      "from": "human",
      "value": "What's in this image?"
    },
    {
      "from": "gpt",
      "value": "The image shows a golden retriever playing in a park..."
    }
  ]
}
```

In [ ]:
def create_instruction_dataset():
    """Create vision-language instruction dataset."""
    
    instructions = [
        {
            "image_id": "img_001.jpg",
            "conversations": [
                {"from": "human", "value": "Describe this image."},
                {"from": "gpt", "value": "A photo of a golden retriever playing fetch in a sunny park."}
            ]
        },
        {
            "image_id": "img_002.jpg",
            "conversations": [
                {"from": "human", "value": "What color is the car?"},
                {"from": "gpt", "value": "The car is red."}
            ]
        },
        {
            "image_id": "img_003.jpg",
            "conversations": [
                {"from": "human", "value": "How many people are in the image?"},
                {"from": "gpt", "value": "There are 3 people in the image."}
            ]
        },
        {
            "image_id": "img_004.jpg",
            "conversations": [
                {"from": "human", "value": "Is there food in this image?"},
                {"from": "gpt", "value": "Yes, there is a pizza on the table."}
            ]
        },
        {
            "image_id": "img_005.jpg",
            "conversations": [
                {"from": "human", "value": "What is the main subject doing?"},
                {"from": "gpt", "value": "The main subject is reading a book while sitting on a bench."}
            ]
        }
    ]
    
    return instructions

# Display instruction dataset
instructions = create_instruction_dataset()

print("Vision-Language Instruction Dataset")
print("="*70)

for i, example in enumerate(instructions, 1):
    print(f"\nExample {i}:")
    print(f"  Image: {example['image_id']}")
    for conv in example['conversations']:
        role = conv['from'].upper()
        value = conv['value']
        print(f"  {role}: {value}")

print("\n" + "="*70)
print("Training Process:")
print("="*70)
print("\n1. STAGE 1: Pre-training Projection")
print("   • Freeze vision encoder and LLM")
print("   • Train only projection layer")
print("   • Dataset: Image-caption pairs (e.g., LAION)")
print("   • Loss: Language modeling loss")
print("   • Duration: ~1 day on 8 GPUs")

print("\n2. STAGE 2: Instruction Tuning")
print("   • Unfreeze LLM (or use LoRA)")
print("   • Keep vision encoder frozen")
print("   • Dataset: Instruction-following examples")
print("   • Loss: Language modeling on responses only")
print("   • Duration: ~1 day on 8 GPUs")

print("\nDataset Statistics:")
print("-"*70)
print("  LLaVA-Instruct-150K: 150,000 instruction examples")
print("  • Conversation: 58K")
print("  • Detail description: 23K")
print("  • Complex reasoning: 77K")

# Visualize instruction types
fig, ax = plt.subplots(figsize=(10, 6))

categories = ['Conversation', 'Detail Description', 'Complex Reasoning']
counts = [58000, 23000, 77000]
colors = ['skyblue', 'lightgreen', 'lightcoral']

bars = ax.bar(categories, counts, color=colors, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Number of Examples', fontsize=12)
ax.set_title('LLaVA Instruction Dataset Composition', fontsize=14, weight='bold')
ax.set_ylim(0, max(counts) * 1.2)

for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1000,
            f'{count:,}', ha='center', va='bottom', fontsize=11, weight='bold')

plt.tight_layout()
plt.show()

## 6. Production Multimodal Pipeline

Building production-ready multimodal systems.

In [ ]:
class ProductionMultimodalPipeline:
    """
    Production pipeline for multimodal tasks.
    """
    
    def __init__(self, model_name: str = "llava-v1.5-7b"):
        self.model_name = model_name
        print(f"Initializing {model_name}...")
        
        # In production, load real model:
        # from llava.model import LlavaLlamaForCausalLM
        # self.model = LlavaLlamaForCausalLM.from_pretrained(model_name)
        
    def process_image(self, image_path: str) -> torch.Tensor:
        """Load and preprocess image."""
        # In production:
        # from PIL import Image
        # image = Image.open(image_path)
        # image = self.processor(image)
        
        # Simulated
        return torch.randn(3, 224, 224)
    
    def generate_caption(self, image_path: str) -> str:
        """Generate caption for image."""
        image = self.process_image(image_path)
        
        # In production:
        # prompt = "Describe this image in detail."
        # output = self.model.generate(image, prompt)
        
        # Simulated
        return "A beautiful sunset over the ocean with vibrant orange and pink colors."
    
    def answer_question(self, image_path: str, question: str) -> str:
        """Answer question about image."""
        image = self.process_image(image_path)
        
        # In production:
        # output = self.model.generate(image, question)
        
        # Simulated
        return f"Based on the image, the answer to '{question}' is..."
    
    def batch_process(self, image_paths: List[str], task: str = "caption") -> List[str]:
        """Process multiple images in batch."""
        results = []
        
        for path in image_paths:
            if task == "caption":
                result = self.generate_caption(path)
            else:
                result = f"Processed {path}"
            
            results.append(result)
        
        return results

# Create pipeline
pipeline = ProductionMultimodalPipeline(model_name="llava-v1.5-7b")

print("\nProduction Multimodal Pipeline")
print("="*70)

# Single image processing
print("\nTask 1: Image Captioning")
caption = pipeline.generate_caption("sample_image.jpg")
print(f"Caption: {caption}")

# VQA
print("\nTask 2: Visual Question Answering")
question = "What color is the sky?"
answer = pipeline.answer_question("sample_image.jpg", question)
print(f"Question: {question}")
print(f"Answer: {answer}")

# Batch processing
print("\nTask 3: Batch Processing")
image_paths = [f"image_{i}.jpg" for i in range(5)]
captions = pipeline.batch_process(image_paths, task="caption")
print(f"Processed {len(captions)} images")

print("\n" + "="*70)
print("Deployment Best Practices:")
print("="*70)

practices = [
    ("Model Selection", "Choose appropriate model size (7B, 13B, 34B) based on latency requirements"),
    ("Quantization", "Use 4-bit/8-bit quantization to reduce memory (GPTQ, AWQ)"),
    ("Batching", "Batch multiple requests for better throughput"),
    ("Caching", "Cache vision embeddings for repeated images"),
    ("Preprocessing", "Resize images to model input size (e.g., 336x336 for LLaVA)"),
    ("Error Handling", "Gracefully handle invalid images, timeouts"),
    ("Monitoring", "Track latency, throughput, error rates"),
    ("A/B Testing", "Compare different models/prompts for your use case")
]

for i, (category, practice) in enumerate(practices, 1):
    print(f"\n{i}. {category}")
    print(f"   → {practice}")

## Summary

You've mastered advanced multimodal applications:
- ✅ LLaVA architecture (vision encoder + projection + LLM)
- ✅ Image captioning (encoder-decoder, generation)
- ✅ Visual Question Answering (image + question → answer)
- ✅ Multimodal RAG (retrieve images and text together)
- ✅ Vision-language instruction tuning (two-stage training)
- ✅ Production pipelines (batching, caching, deployment)

**Key Insights**:
1. **LLaVA** connects frozen vision encoder to LLM via trainable projection
2. **Two-stage training** first aligns vision-language, then fine-tunes for instructions
3. **Image captioning** uses encoder-decoder architecture with autoregressive generation
4. **VQA** fuses visual and textual features for classification or generation
5. **Multimodal RAG** retrieves relevant images and text for context-aware answers
6. **Production deployment** requires quantization, batching, and caching

**Model Comparison**:
| Model | Size | Strengths | Use Case |
|-------|------|-----------|----------|
| CLIP | 400M | Zero-shot classification | Search, retrieval |
| LLaVA | 7B-13B | Instruction following | General VQA, captioning |
| GPT-4V | ~1T | State-of-the-art | Premium applications |
| Flamingo | 80B | Few-shot learning | Research |
| BLIP-2 | 2.7B | Efficient VLMs | Resource-constrained |

**Applications**:
- **E-commerce**: Product search, visual recommendations
- **Healthcare**: Medical image analysis, report generation
- **Education**: Interactive learning, image explanations
- **Accessibility**: Image descriptions for visually impaired
- **Content moderation**: Detect inappropriate content
- **Document analysis**: Extract information from diagrams

**Future Directions**:
- Video understanding (temporal reasoning)
- 3D vision-language models
- Multi-turn visual dialogue
- Grounding (link text to image regions)
- Video generation from text

**Practical Tips**:
1. Start with CLIP for simple tasks (classification, retrieval)
2. Use LLaVA for instruction-following VQA
3. Fine-tune on domain-specific data for best results
4. Combine with RAG for knowledge-intensive tasks
5. Monitor hallucinations in generated descriptions

## Exercises

1. **Fine-tune LLaVA**: Adapt LLaVA to medical images with LoRA
2. **Multi-turn dialogue**: Build conversational VQA system
3. **Visual grounding**: Highlight image regions relevant to answer
4. **Video QA**: Extend VQA to video understanding